<a href="https://colab.research.google.com/github/kostya27111984-ai/Simulative/blob/main/Sim_1_18_Case_2_%22%D0%90%D0%BA%D1%82%D0%B8%D0%B2%D0%BD%D0%BE%D1%81%D1%82%D1%8C_Itresume%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **2 кейс**

**Выгрузка активности с ItResume**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [1]:
!wget https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv

--2026-05-15 07:04:59--  https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.github.com (gist.github.com)... 140.82.112.4
Connecting to gist.github.com (gist.github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv [following]
--2026-05-15 07:04:59--  https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215378 (210K) [text/plain]
Saving to: ‘codesubmit.csv’

codesubmit.csv      100%[===================>] 210.33K  --.-KB/s    in 0.02s   

2026-05-15 07:04:59 (8.76 MB/s) - ‘co

Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [ ]:
import pandas as pd

df = pd.read_csv('codesubmit.csv', sep = ';')
df

,created_at,user_id,problem_id,is_correct,type
0,2023-04-30 13:47:14.344471,7,870,1.0,submit
1,2023-04-30 13:46:15.949925,7,870,0.0,submit
2,2023-04-30 16:13:26.005286,173,21,1.0,submit
3,2023-04-30 16:13:06.739782,173,21,NaN,run
4,2023-04-30 15:52:00.195532,173,25,1.0,submit
...,...,...,...,...,...
4994,2023-04-30 21:52:00.269123,13493,435,NaN,run
4995,2023-04-30 21:51:01.094234,13493,435,1.0,submit
4996,2023-04-30 21:50:52.059690,13493,435,NaN,run
4997,2023-04-30 21:42:24.323689,13493,1086,NaN,run


### **Решения**

#### **Задача 1**

Ваша задача - выяснить сколько в среднем тратится времени на решение задачи.

**Примечание**: для правильного подсчета - рассчитайте сначала среднее время решения по каждой задаче в отдельности, и только затем находите общее среднее время решения задач.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res`.


**Решение**

Напишите свое решение ниже

In [51]:
import csv
from datetime import datetime
from statistics import mean
from collections import defaultdict

s = []
tasks = set()
with open('codesubmit.csv', newline='') as csvfile:
    reader = csv.reader(csvfile, delimiter=';', quotechar='"')
    next(reader)
    for row in reader:
        temp = row
        f = row[2]+'_'+row[1]
        temp.append(f)
        tasks.add(f)
        s.append(temp)

# ИЗМЕНЕНИЕ 1: defaultdict(list) вместо обычного dict
time_dict = defaultdict(list)

for i in tasks:
    start = None
    end = None
    for j in s:
        if i == j[5]:
            temp_start = datetime.strptime(j[0], '%Y-%m-%d %H:%M:%S.%f')
            if start is None or temp_start < start:
              start = temp_start
        if i == j[5] and j[4] == 'submit' and j[3] == '1':
            temp_end = datetime.strptime(j[0], '%Y-%m-%d %H:%M:%S.%f')
            if end and temp_end < end:
                end = temp_end
            if not end:
                end = temp_end
    if start and end and start < end:
        # ИЗМЕНЕНИЕ 2: группируем по problem_id и добавляем в список
        problem_id = i.split('_')[0]
        time_dict[problem_id].append((end - start).total_seconds())

# ИЗМЕНЕНИЕ 3: сначала среднее по задаче
avg_per_problem = {p: mean(times) for p, times in time_dict.items()}

# ИЗМЕНЕНИЕ 4: затем общее среднее
res = round(mean(avg_per_problem.values()), 2)
print(res)

611.86


In [47]:
#Оптимизированный код
import csv
from datetime import datetime
from statistics import mean
from collections import defaultdict

s = []
with open('codesubmit.csv', newline='') as csvfile:
    reader = csv.reader(csvfile, delimiter=';', quotechar='"')
    next(reader)  # пропускаем заголовок
    for row in reader:
        s.append(row)

# Группируем по (problem_id, user_id)
tasks = defaultdict(list)
for (problem_id, user_id) in set((row[2], row[1]) for row in s):
    events = [r for r in s if r[2] == problem_id and r[1] == user_id]
    events.sort(key=lambda x: datetime.strptime(x[0], '%Y-%m-%d %H:%M:%S.%f'))

    first_attempt = datetime.strptime(events[0][0], '%Y-%m-%d %H:%M:%S.%f')

    # Первый успешный submit
    success = [e for e in events if e[4] == 'submit' and e[3] == '1']
    if not success:
        continue

    first_success = datetime.strptime(success[0][0], '%Y-%m-%d %H:%M:%S.%f')

    # Исключаем гениев
    if first_attempt == first_success:
        continue

    time_diff = (first_success - first_attempt).total_seconds()
    tasks[problem_id].append(time_diff)

# Среднее по задаче, затем общее
avg_per_task = {pid: mean(times) for pid, times in tasks.items()}
res = round(mean(avg_per_task.values()), 2)
res


611.86

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [52]:
try:
    assert res == 611.86
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача - выяснить сколько часов в среднем проводит юзер в день на платформе. Перерывы в активности за день - не учитываем.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res2`.

**Решение**

Напишите свое решение ниже

In [66]:
import csv
from datetime import datetime
from statistics import mean

s = []
with open('codesubmit.csv', newline='') as csvfile:
    reader = csv.reader(csvfile, delimiter=';', quotechar='"')
    next(reader)
    for row in reader:
        s.append(row)

threshold_minutes = 300  # 5 часов

user_avg_times = []

for user_id in set(row[1] for row in s):
    events = [r for r in s if r[1] == user_id]
    events_dt = [datetime.strptime(e[0], '%Y-%m-%d %H:%M:%S.%f') for e in events]
    events_dt.sort()

    dates = set([dt.date() for dt in events_dt])

    daily_times = []
    for date in dates:
        day_events = [dt for dt in events_dt if dt.date() == date]

        if len(day_events) < 2:
            continue

        # Суммируем только интервалы <= 5 часов
        total_time = 0
        for i in range(len(day_events) - 1):
            interval = (day_events[i+1] - day_events[i]).total_seconds() / 60
            if interval <= threshold_minutes:
                total_time += interval

        if total_time > 0:
            daily_times.append(total_time / 60)

    if daily_times:
        user_avg_times.append(mean(daily_times))

res2 = round(mean(user_avg_times), 2)
print(res2)

1.7


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [67]:
try:
    assert res2 == 1.7
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 3**

Теперь давайте посмотрим на активные сеансы. Выясните, сколько задач в среднем решается за один активный сеанс.

**Активный сеанс** - период, когда между любой активностью пользователя разница менее или равна часу, не более

**Важно**: в расчет берем не только успешные попытки решений (`is_correct=1`), а и неуспешные тоже (`is_correct=0`), и тип `run` в том числе.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res3`.

**Решение**

Напишите свое решение ниже

In [83]:
import csv
from datetime import datetime
from statistics import mean

s = []
with open('codesubmit.csv', newline='') as csvfile:
    reader = csv.reader(csvfile, delimiter=';', quotechar='"')
    next(reader)
    for row in reader:
        s.append(row)

all_sessions = []

for user_id in set(row[1] for row in s):
    events = [r[:] for r in s if r[1] == user_id]
    for r in events:
      r[0] = datetime.strptime(r[0], '%Y-%m-%d %H:%M:%S.%f')
    events.sort(key=lambda x: x[0])

    if len(events) < 1:
      continue
    else:
      temp_date = events[0][0]
      t = [events[0][2]]

      for i in events[1:]:
        length = (i[0] - temp_date).total_seconds()/3600
        if  length <= 1:
          t.append(i[2])
        else:
          all_sessions.append(len(set(t)))
          t = [i[2]]
        temp_date = i[0]
      all_sessions.append(len(set(t)))

res3 =round(mean(all_sessions), 2)
res3

3.14

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [84]:
try:
    assert res3 == 3.14
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 4**

И финальная - найдите самый "популярный" час дня на нашей платформе.

Популярность определяем максимальным количеством уникальных пользователей, совершающих какую-либо активность в этот период

Результат в числовом формате запишите в переменную `res4`.

Например, самым популярным часом стал период с 22 до 23, тогда в переменной `res4` должно лежать **22**. Обозначающее начало этого периода.

**Решение**

Напишите свое решение ниже

In [86]:
import csv
from datetime import datetime
from collections import defaultdict

s = []
with open('codesubmit.csv', newline='') as csvfile:
    reader = csv.reader(csvfile, delimiter=';', quotechar='"')
    next(reader)
    for row in reader:
        s.append(row)

# Группируем по часам: для каждого часа собираем уникальных пользователей
hour_users = defaultdict(set)

for row in s:
    dt = datetime.strptime(row[0], '%Y-%m-%d %H:%M:%S.%f')
    hour = dt.hour  # 0-23
    user_id = row[1]
    hour_users[hour].add(user_id)

# Находим час с максимальным количеством уникальных пользователей
max_hour = max(hour_users.items(), key=lambda x: len(x[1]))[0]

res4 = max_hour
print(res4)


16


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [87]:
try:
    assert res4 == 16
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
